# 02 — Cleaning and preprocessing

Parse the two observed timestamp formats, canonicalize observed transaction type variants, retain rows, derive time and balance features, and fit imputation/encoding/scaling only on the training partition. The raw file remains unchanged.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
ROOT=Path.cwd()
if not (ROOT/'data_science').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'data_science'))
from cleaning import clean_transactions, make_model_features
raw=pd.read_csv(ROOT/'messy_fraud_dataset_20k.csv',low_memory=False)
data=clean_transactions(raw)
data.shape, data.isna().sum()

In [ ]:
X=make_model_features(data)
y=data.is_fraud.astype(int)
groups=data.customer_id
outer=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=42)
train_idx,hold_idx=next(outer.split(X,y,groups))
hold_X,hold_y,hold_g=X.iloc[hold_idx],y.iloc[hold_idx],groups.iloc[hold_idx]
inner=StratifiedGroupKFold(n_splits=2,shuffle=True,random_state=43)
valid_rel,test_rel=next(inner.split(hold_X,hold_y,hold_g))
valid_idx,test_idx=hold_idx[valid_rel],hold_idx[test_rel]
[(len(idx),int(y.iloc[idx].sum())) for idx in (train_idx,valid_idx,test_idx)]

In [ ]:
assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[valid_idx])
assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[test_idx])
assert set(groups.iloc[valid_idx]).isdisjoint(groups.iloc[test_idx])
# Class weights handle imbalance; never oversample before splitting.
# Identifier columns txn_id/customer_id/device_ip and target is_fraud are absent from X.
# The model training entry point fits all imputers/encoders/scalers on training rows only.
X.columns.tolist()